# Analisis RFM

Notebook de exploracion y validacion para la capa `rfm__segmentacion_clientes__v1.parquet`.

## Objetivos

- validar la salida del pipeline RFM;
- revisar la distribucion de segmentos;
- extraer hallazgos reutilizables para README y dashboard.

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
processed_dir = ROOT / 'data' / 'processed'

rfm = pd.read_parquet(processed_dir / 'rfm__segmentacion_clientes__v1.parquet')
clientes = pd.read_parquet(processed_dir / 'clientes__base_analitica__v1.parquet')

rfm.head()

,customer_id,snapshot_datetime,snapshot_date,first_purchase_datetime,last_purchase_datetime,recency_days,frequency_orders,monetary_gbp,r_score,f_score,m_score,rfm_score,rfm_segment
0,12346,2011-12-09 12:50:00,2011-12-09,2009-12-14 08:34:00,2011-01-18 10:01:00,325,12,77556.46,2,5,5,255,at_risk
1,12347,2011-12-09 12:50:00,2011-12-09,2010-10-31 14:20:00,2011-12-07 15:52:00,2,8,5633.32,5,4,5,545,champions
2,12348,2011-12-09 12:50:00,2011-12-09,2010-09-27 14:59:00,2011-09-25 13:13:00,75,5,2019.4,3,4,4,344,loyal_customers
3,12349,2011-12-09 12:50:00,2011-12-09,2010-04-29 13:20:00,2011-11-21 09:51:00,18,4,4428.69,5,3,5,535,potential_loyalists
4,12350,2011-12-09 12:50:00,2011-12-09,2011-02-02 16:01:00,2011-02-02 16:01:00,310,1,334.4,2,1,2,212,hibernating


In [2]:
rfm['rfm_segment'].value_counts(dropna=False).to_frame('n_customers')

,n_customers
rfm_segment,
hibernating,2372
champions,1270
at_risk,992
potential_loyalists,662
loyal_customers,582


In [3]:
segment_summary = (
    rfm.groupby('rfm_segment', as_index=False)
    .agg(
        n_customers=('customer_id', 'size'),
        revenue_total_gbp=('monetary_gbp', 'sum'),
        avg_recency_days=('recency_days', 'mean'),
        avg_frequency_orders=('frequency_orders', 'mean'),
    )
    .sort_values('revenue_total_gbp', ascending=False)
)
segment_summary

,rfm_segment,n_customers,revenue_total_gbp,avg_recency_days,avg_frequency_orders
1,champions,1270,12029492.755,19.188976,17.366929
0,at_risk,992,1938270.363,376.420363,4.344758
3,loyal_customers,582,1791472.675,86.343643,8.43299
2,hibernating,2372,1059172.184,301.967116,1.564503
4,potential_loyalists,662,867052.661,24.770393,2.996979


In [4]:
segment_summary.assign(
    revenue_share=lambda df: df['revenue_total_gbp'] / df['revenue_total_gbp'].sum(),
    customer_share=lambda df: df['n_customers'] / df['n_customers'].sum(),
)

,rfm_segment,n_customers,revenue_total_gbp,avg_recency_days,avg_frequency_orders,revenue_share,customer_share
1,champions,1270,12029492.755,19.188976,17.366929,0.680191,0.21606
0,at_risk,992,1938270.363,376.420363,4.344758,0.109597,0.168765
3,loyal_customers,582,1791472.675,86.343643,8.43299,0.101296,0.099013
2,hibernating,2372,1059172.184,301.967116,1.564503,0.059889,0.403539
4,potential_loyalists,662,867052.661,24.770393,2.996979,0.049026,0.112623


In [5]:
(
    clientes[['customer_id', 'primary_country', 'is_repeat_customer', 'total_revenue_gbp']]
    .merge(rfm[['customer_id', 'rfm_segment']], on='customer_id', how='inner')
    .groupby(['primary_country', 'rfm_segment'], as_index=False)
    .agg(
        n_customers=('customer_id', 'size'),
        revenue_total_gbp=('total_revenue_gbp', 'sum'),
    )
    .sort_values(['primary_country', 'revenue_total_gbp'], ascending=[True, False])
    .head(20)
)

,primary_country,rfm_segment,n_customers,revenue_total_gbp
1,Australia,champions,3,159149.95
2,Australia,hibernating,8,6181.41
3,Australia,loyal_customers,1,2399.45
0,Australia,at_risk,2,2094.13
5,Austria,champions,2,8139.96
4,Austria,at_risk,6,6963.47
7,Austria,potential_loyalists,2,6312.69
6,Austria,hibernating,1,707.09
8,Bahrain,at_risk,1,947.61
9,Bahrain,hibernating,1,406.76


## Hallazgos esperados

- `champions` debe concentrar una parte relevante del revenue total.
- `hibernating` suele ser el segmento mas grande en volumen.
- `at_risk` ayuda a detectar clientes de valor historico con baja recencia.